In [5]:
import json
import pandas as pd
from itertools import chain
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from datetime import datetime
import re
from functools import reduce
from operator import mul

ru_months = {
    1: 'января', 2: 'февраля', 3: 'марта', 4: 'апреля', 
    5: 'мая', 6: 'июня', 7: 'июля', 8: 'августа',
    9: 'сентября', 10: 'октября', 11: 'ноября', 12: 'декабря'
}

pass_mapping_0 = {
    "adults": "взрослых",
    "children": "детей",
    "infants": "младенцев"
}

pass_mapping_1 = {
    "adults": "взрослого",
    "children": "ребенка",
    "infants": "младенца"
}

_PC_RE = re.compile(r"1PC(\d+)(?:x(\d+)x(\d+)x(\d+))?$")

def minimal_baggage_limits(matrix):

    # helper: detect any falsy item anywhere in the structure
    def has_falsy(node):
        if isinstance(node, list):
            return any(has_falsy(x) for x in node)
        return not node

    if has_falsy(matrix):
        return ""

    # flatten, parse and bucketise
    weight_only, weight_dims = [], []
    def collect(node):
        for x in node:
            if isinstance(x, list):
                collect(x)
            else:
                m = _PC_RE.fullmatch(str(x))
                if m:
                    w = int(m.group(1))
                    if m.group(2):
                        dims = tuple(map(int, m.groups()[1:]))
                        weight_dims.append((w, dims, x))
                    else:
                        weight_only.append((w, x))
    collect(matrix)

    # case 3 – weight‑only tokens exist
    if weight_only:
        min_w = min(w for w, _ in weight_only)
        return next(s for w, s in weight_only if w == min_w)

    # case 4 – only weight+dim tokens
    if weight_dims:
        min_w = min(w for w, *_ in weight_dims)
        # choose the one with the smallest volume
        best = min(
            ((reduce(mul, dims), s) for w, dims, s in weight_dims if w == min_w),
            key=lambda t: t[0],
        )[1]
        return best

    return ""  # nothing parsable

def format_date_russian(date_str):
    date_obj = datetime.strptime(date_str, "%Y-%m-%d")
    
    day = date_obj.day
    month = ru_months[date_obj.month]
    year = date_obj.year
    
    return f"{day} {month}"

def format_seconds(seconds):
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    days, hours = divmod(hours, 24)
    
    if days > 0:
        return f"{days}дн {hours}ч {minutes}мин"
    else:
        return f"{hours}ч {minutes}мин"
    
def format_passengers(num, pass_type):
    if num % 10 == 1:
        return f"{num} {pass_mapping_1[pass_type]}"
    else:
        return f"{num} {pass_mapping_0[pass_type]}"
    
def summarize_exchange_return(fares_a, fares_b, mode="functional"):
    fares_chain = list(chain(*(fares_a, fares_b)))
    def restricted(key):
        for d in fares_chain:
            if not isinstance(d, dict) or key not in d or "available" not in d[key] or d[key]["available"] is False:
                return False
        return True

    if mode == "get_str":
        if not fares_chain:
            return "без обмена, без возврата"
        has_change = restricted("change_before_flight")
        has_refund = restricted("return_before_flight")
        part_change = "с обменом" if has_change else "без обмена"
        part_refund = "с возвратом" if has_refund else "без возврата"
        return f"{part_change}, {part_refund}"
    return restricted("change_before_flight"), restricted("return_before_flight")

def round_price_k(price, num):
    return round(price / num) * num

with open("../digital_assistant_first/aviasales_system/search_json_received.json", "r") as f:
    res = json.load(f)

with open("../digital_assistant_first/aviasales_system/aviasales_json_sent.json", "r") as f:
    AVIASALES_JSON = json.load(f)

with open("../digital_assistant_first/aviasales_system/iata_airports.json", "r") as f:
    airports_mapping = pd.DataFrame(json.load(f))

with open("../digital_assistant_first/aviasales_system/iata_cities.json", "r") as f:
    cities_mapping = pd.DataFrame(json.load(f))

with open("../digital_assistant_first/aviasales_system/iata_airlines.json", "r") as f:
    airlines_mapping = pd.DataFrame(json.load(f))

airports_mapping.name = airports_mapping.apply(
    lambda x: x["name"] if x["name"] else x["name_translations"]["en"]
, axis=1)
airlines_mapping.name = airlines_mapping.apply(
    lambda x: x["name"] if x["name"] else x["name_translations"]["en"]
, axis=1)

cities_mapping.set_index("code", inplace=True)
airports_mapping.set_index("code", inplace=True)
airlines_mapping.set_index("code", inplace=True)

cities_mapping = cities_mapping[["name"]].copy()
airports_mapping = airports_mapping[["name", "city_code"]].copy()
airlines_mapping = airlines_mapping[["name", "is_lowcost"]].copy()

# AVIASALES_JSON = {'origin': 'SGC', 'destination': 'UUA', 'adults': 1, 'children': 0, 'infants': 0, 'baggage': False, 'to': '2025-05-07', 'back': '2025-05-14', 'travel_class': 'Y', 'airlines': ['U6', 'SU', 'EY', 'EK', 'S7', 'U2'], 'max_stops': 2, 'to_hour_range': '18-24', 'from_hour_range': '00-12'}

In [6]:
proposals = list(chain(*[i.get("proposals", []) for i in res]))
proposals_df = pd.DataFrame(proposals)
proposals_df["pricing_info"] = proposals_df.xterms.apply(
    lambda x: [
        {k: v for k, v in i.items() if k not in ["baggage_source", "handbags_source"]}
        for i in list(list(x.values())[0].values())
    ]
)

In [7]:
proposals_df.columns

Index(['terms', 'xterms', 'segment', 'total_duration', 'stops_airports',
       'is_charter', 'max_stops', 'max_stop_duration', 'min_stop_duration',
       'carriers', 'segment_durations', 'segments_time', 'segments_airports',
       'sign', 'is_direct', 'flight_weight', 'popularity', 'segments_rating',
       'tags', 'validating_carrier', 'pricing_info'],
      dtype='object')

In [8]:
pd.set_option("display.max_columns", None)
proposals_df.drop(
    columns=[
        "terms",
        "xterms",
        "sign",
        "flight_weight",
        "validating_carrier",
        "min_stop_duration",
        "is_charter",
        "is_direct",
        "popularity",
    ],
    inplace=True,
)

proposals_df["idx"] = proposals_df.index
proposals_df_full = proposals_df.explode("pricing_info").reset_index(drop=True)
proposals_df_full["currency"] = proposals_df_full.pricing_info.apply(lambda x: x["currency"])
proposals_df_full["price"] = proposals_df_full.pricing_info.apply(lambda x: x["price"])
proposals_df_full["url"] = proposals_df_full.pricing_info.apply(lambda x: x["url"])
proposals_df_full["flights_baggage"] = proposals_df_full.pricing_info.apply(lambda x: x["flights_baggage"])
proposals_df_full["flights_handbags"] = proposals_df_full.pricing_info.apply(lambda x: x["flights_handbags"])
proposals_df_full["flight_additional_tariff_infos"] = proposals_df_full.pricing_info.apply(lambda x: x["flight_additional_tariff_infos"])
proposals_df_full.drop(columns=["pricing_info"], inplace=True)

proposals_df_full["viable_handbags"] = proposals_df_full.flights_handbags.apply(
    minimal_baggage_limits
)

proposals_df_full["viable_baggage"] = proposals_df_full.flights_baggage.apply(
    minimal_baggage_limits
)

proposals_df_full["has_handbags"] = proposals_df_full.viable_handbags.apply(lambda x: x != "")
proposals_df_full["has_baggage"] = proposals_df_full.viable_baggage.apply(lambda x: x != "")

proposals_df_full["flight_to"] = proposals_df_full.segment.apply(lambda x: x[0])
proposals_df_full["flight_back"] = proposals_df_full.segment.apply(lambda x: x[1] if len(x) > 1 else None)

proposals_df_full["duration_to"] = proposals_df_full.segment_durations.apply(lambda x: x[0])
proposals_df_full["duration_back"] = proposals_df_full.segment_durations.apply(lambda x: x[1] if x else None)

proposals_df_full["tariff_to"] = proposals_df_full.flight_additional_tariff_infos.apply(lambda x: x[0])
proposals_df_full["tariff_back"] = proposals_df_full.flight_additional_tariff_infos.apply(lambda x: x[1] if x else None)

proposals_df_full["rounded_price_5k"] = proposals_df_full["price"].apply(lambda x: round_price_k(x, 10_000)) # chngd from 5k

proposals_df_full["has_exchange"] = proposals_df_full.apply(
    lambda row: summarize_exchange_return(row["tariff_to"], row["tariff_back"])
    if row["tariff_back"]
    else summarize_exchange_return(row["tariff_to"], row["tariff_to"]),
    axis=1,
)
proposals_df_full["has_return"] = proposals_df_full.has_exchange.apply(lambda x: x[1])
proposals_df_full["has_exchange"] = proposals_df_full.has_exchange.apply(lambda x: x[0])
# proposals_df_full["rounded_price_10k"] = proposals_df_full["price"].apply(lambda x: round_price_k(x, 10_000))

proposals_df_full["flight_to_time"] = proposals_df_full.flight_to.apply(lambda x: int(x["flight"][0]["departure_time"].split(":")[0]))
proposals_df_full["flight_back_time"] = proposals_df_full.flight_back.apply(lambda x: int(x["flight"][0]["departure_time"].split(":")[0]))

In [9]:
proposals_df_full.max_stops

0       2
1       2
2       2
3       2
4       2
       ..
6092    2
6093    2
6094    2
6095    2
6096    2
Name: max_stops, Length: 6097, dtype: int64

In [5]:
proposals_df_full.sample(5)

,segment,total_duration,stops_airports,max_stops,max_stop_duration,carriers,segment_durations,segments_time,segments_airports,segments_rating,tags,idx,currency,price,url,flights_baggage,flights_handbags,flight_additional_tariff_infos,viable_handbags,viable_baggage,has_handbags,has_baggage,flight_to,flight_back,duration_to,duration_back,tariff_to,tariff_back,rounded_price_5k,has_exchange,has_return,flight_to_time,flight_back_time
1792,[{'flight': [{'aircraft': 'Sukhoi Superjet 100...,375,"[TBS, VKO]",0,0,"[WZ, A4]","[190, 185]","[[1746606000, 1746621000], [1747247700, 174725...","[[ZIA, TBS], [TBS, VKO]]",9.264778,"[convenient_ticket, direct]",1446,rub,45710,18300424,"[[1PC20], [1PC23]]","[[1PC5x30x20x40], [1PC5x40x20x55]]",[[{'return_before_flight': {'available': False...,1PC5x30x20x40,1PC20,True,True,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,{'flight': [{'aircraft': 'Sukhoi Superjet SU-1...,190,185,[{'return_before_flight': {'available': False}...,[{'return_before_flight': {'available': False}...,50000,False,False,8,18
1915,[{'flight': [{'aircraft': 'Airbus A320-100/200...,1870,"[GYD, TBS, GYD, DME]",1,885,[J2],"[710, 1160]","[[1746622200, 1746668400], [1747241400, 174730...","[[VKO, TBS], [TBS, DME]]",5.977333,"[overnight_layover, night_transfer, long_layov...",1512,rub,61529,18300561,"[[False, False], [False, False]]","[[1PC10x40x23x55, 1PC10x40x23x55], [1PC10x40x2...",[[{'return_before_flight': {'available': False...,1PC10x40x23x55,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Embraer EMB 190 / EM...,710,1160,[{'return_before_flight': {'available': False}...,[{'return_before_flight': {'available': False}...,60000,False,False,12,16
2606,[{'flight': [{'aircraft': 'Boeing 737-800 (win...,1840,"[DXB, TBS, DXB, VKO]",1,605,[FZ],"[700, 1140]","[[1746579900, 1746625500], [1747202700, 174726...","[[VKO, TBS], [TBS, VKO]]",6.070667,[long_layover],1851,rub,111455,18301167,"[[, ], [1PC30, 1PC30]]","[[1PC7x38x20x55, 1PC7x38x20x55], [1PC7x38x20x5...",[[{'return_before_flight': {'available': False...,1PC7x38x20x55,,True,False,{'flight': [{'aircraft': 'Boeing 737-800 (wing...,"{'flight': [{'aircraft': 'Boeing 737 Max 8', '...",700,1140,[{'return_before_flight': {'available': False}...,"[{'return_before_flight': {'available': True, ...",110000,False,False,1,6
1098,"[{'flight': [{'aircraft': 'Boeing 737-800', 'a...",1725,"[MRV, TBS, MRV, SVO]",1,770,"[DP, WZ, SU]","[705, 1020]","[[1746612000, 1746657900], [1747266300, 174732...","[[VKO, TBS], [TBS, SVO]]",6.034333,"[night_transfer, long_layover, overnight_layov...",1042,rub,38705,6500652,"[[False, False], [False, False]]","[[1PC10x27x30x36, 1PC5x30x20x40], [1PC5x30x20x...",[[{'return_before_flight': {'available': False...,1PC5x30x20x40,,True,False,"{'flight': [{'aircraft': 'Boeing 737-800', 'ar...",{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,705,1020,[{'return_before_flight': {'available': False}...,[{'return_before_flight': {'available': False}...,40000,True,False,10,23
1722,[{'flight': [{'aircraft': 'Sukhoi Superjet SU-...,745,"[TBS, AER, DME]",1,270,"[A4, WZ, U6]","[195, 550]","[[1746624000, 1746639300], [1747215300, 174724...","[[VKO, TBS], [TBS, DME]]",7.061988,"[convenient_ticket, long_layover]",1412,rub,42764,18300356,"[[1PC23], [1PC20, 1PC23]]","[[1PC5x40x20x55], [1PC5x30x20x40, 1PC10x40x20x...",[[{'return_before_flight': {'available': False...,1PC5x30x20x40,1PC20,True,True,{'flight': [{'aircraft': 'Sukhoi Superjet SU-1...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,195,550,[{'return_before_flight': {'available': False}...,[{'return_before_flight': {'available': False}...,40000,False,False,13,9


In [6]:
AVIASALES_JSON

{'origin': 'MOW',
 'destination': 'TBS',
 'adults': 1,
 'children': 0,
 'infants': 0,
 'handbag': True,
 'baggage': False,
 'to': '2025-05-07',
 'back': '2025-05-14',
 'travel_class': 'Y',
 'airlines': [],
 'blacklist_airlines': [],
 'max_stops': 2,
 'to_hour_range': '18-24',
 'from_hour_range': '00-12'}

[FILTERS] time preference & luggage & airlines filter:

In [7]:
min_hr_to, max_hr_to = 0, 24
min_hr_back, max_hr_back = 0, 24

if AVIASALES_JSON.get("to_hour_range", None):
    min_hr_to, max_hr_to = list(map(int, AVIASALES_JSON["to_hour_range"].split("-")))
if AVIASALES_JSON.get("from_hour_range", None):
    min_hr_back, max_hr_back = list(map(int, AVIASALES_JSON["from_hour_range"].split("-")))

min_hr_to, max_hr_to, min_hr_back, max_hr_back

(18, 24, 0, 12)

In [8]:
handbag_required = AVIASALES_JSON.get("handbag", False)
baggage_required = AVIASALES_JSON.get("baggage", False)

handbag_required, baggage_required

(True, False)

In [9]:
preferred_airlines = set(AVIASALES_JSON.get("airlines", []))
blacklist_airlines = set(AVIASALES_JSON.get("blacklist_airlines", []))

preferred_airlines, blacklist_airlines

(set(), set())

In [10]:
proposals_df_full["has_blacklisted_airlines"] = proposals_df_full.carriers.apply(
    lambda x: True if len(set(x).intersection(blacklist_airlines)) > 0 else False
)

In [11]:
viable_proposals = proposals_df_full[
    (proposals_df_full.flight_to_time >= min_hr_to) & (proposals_df_full.flight_to_time <= max_hr_to) &
    (proposals_df_full.flight_back_time >= min_hr_back) & (proposals_df_full.flight_back_time <= max_hr_back) &
    (proposals_df_full.has_handbags == handbag_required) & (proposals_df_full.has_baggage == baggage_required) &
    (proposals_df_full.has_blacklisted_airlines == False)
].copy()

viable_proposals_other = proposals_df_full.iloc[~viable_proposals.index]
# viable_proposals.sample(5)
# if shape is 0, then disable baggage filter
# if shape is 0, then tighten time filter

ranking for optimality:

1) sort by rounded_price, get ticket with lowest time & stops (step 2 - sort by orig price)
2) sort by time & stops, get ticket with lowest rounded_price (step 2 - sort by orig price)

optimal (быстрые оптимальные):

In [12]:
indices_already_displayed = []

In [13]:
min_time_values = viable_proposals["total_duration"].nsmallest(2).unique()
fastest_optimal = viable_proposals[viable_proposals["total_duration"].isin(min_time_values)].sort_values("price").iloc[:5]
viable_proposals.drop(fastest_optimal.index, inplace=True)
indices_already_displayed.extend(fastest_optimal.idx.to_list())
fastest_optimal

,segment,total_duration,stops_airports,max_stops,max_stop_duration,carriers,segment_durations,segments_time,segments_airports,segments_rating,tags,idx,currency,price,url,flights_baggage,flights_handbags,flight_additional_tariff_infos,viable_handbags,viable_baggage,has_handbags,has_baggage,flight_to,flight_back,duration_to,duration_back,tariff_to,tariff_back,rounded_price_5k,has_exchange,has_return,flight_to_time,flight_back_time,has_blacklisted_airlines
687,[{'flight': [{'aircraft': 'Airbus A320-100/200...,1795,"[EVN, TBS, AER, SVO]",1,1055,"[3F, WZ, DP]","[1280, 515]","[[1746646200, 1746726600], [1747215300, 174724...","[[DME, TBS], [TBS, SVO]]",6.218333,"[night_transfer, long_layover, overnight_layov...",631,rub,36228,6500243,"[[False, False], [False, False]]","[[1PCx30x20x40, 1PCx30x20x40], [1PC5x30x20x40,...","[[{}, {}], [{'return_before_flight': {'availab...",1PC5x30x20x40,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,1280,515,"[{}, {}]",[{'return_before_flight': {'available': False}...,40000,False,False,19,9,False
945,[{'flight': [{'aircraft': 'Airbus A320-100/200...,1735,"[EVN, TBS, AER, DME]",1,1055,"[3F, WZ, U6]","[1280, 455]","[[1746646200, 1746726600], [1747215300, 174723...","[[DME, TBS], [TBS, DME]]",6.230333,"[long_layover, overnight_layover, night_transf...",889,rub,38041,6500499,"[[False, False], [False, False]]","[[1PCx30x20x40, 1PCx30x20x40], [1PC5x30x20x40,...","[[{}, {}], [{'return_before_flight': {'availab...",1PC5x30x20x40,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,1280,455,"[{}, {}]",[{'return_before_flight': {'available': False}...,40000,False,False,19,9,False


optimal (дешевые оптимальные):

In [14]:
viable_proposals = viable_proposals[~viable_proposals.idx.isin(indices_already_displayed)].copy()
min_price = viable_proposals["rounded_price_5k"].min()
cheapest_optimal = viable_proposals[viable_proposals["rounded_price_5k"] == min_price].sort_values("total_duration").iloc[:5]
viable_proposals.drop(cheapest_optimal.index, inplace=True)
indices_already_displayed.extend(cheapest_optimal.idx.to_list())
cheapest_optimal

,segment,total_duration,stops_airports,max_stops,max_stop_duration,carriers,segment_durations,segments_time,segments_airports,segments_rating,tags,idx,currency,price,url,flights_baggage,flights_handbags,flight_additional_tariff_infos,viable_handbags,viable_baggage,has_handbags,has_baggage,flight_to,flight_back,duration_to,duration_back,tariff_to,tariff_back,rounded_price_5k,has_exchange,has_return,flight_to_time,flight_back_time,has_blacklisted_airlines
568,[{'flight': [{'aircraft': 'Airbus A320-100/200...,2940,"[EVN, TBS, AER, VKO]",1,1370,"[3F, WZ, DP]","[1280, 1660]","[[1746646200, 1746726600], [1747215300, 174731...","[[DME, TBS], [TBS, VKO]]",5.58,"[has_night_transfer, night_transfer, long_layo...",512,rub,34695,6500122,"[[False, False], [False, False]]","[[1PCx30x20x40, 1PCx30x20x40], [1PC5x30x20x40,...","[[{}, {}], [{'return_before_flight': {'availab...",1PC5x30x20x40,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,1280,1660,"[{}, {}]",[{'return_before_flight': {'available': False}...,30000,False,False,19,9,False


дешевые:

In [15]:
viable_proposals = viable_proposals[~viable_proposals.idx.isin(indices_already_displayed)].copy()
cheapest = viable_proposals.sort_values("price").iloc[:5]
viable_proposals.drop(cheapest.index, inplace=True)
indices_already_displayed.extend(cheapest.idx.to_list())
cheapest

,segment,total_duration,stops_airports,max_stops,max_stop_duration,carriers,segment_durations,segments_time,segments_airports,segments_rating,tags,idx,currency,price,url,flights_baggage,flights_handbags,flight_additional_tariff_infos,viable_handbags,viable_baggage,has_handbags,has_baggage,flight_to,flight_back,duration_to,duration_back,tariff_to,tariff_back,rounded_price_5k,has_exchange,has_return,flight_to_time,flight_back_time,has_blacklisted_airlines
686,[{'flight': [{'aircraft': 'Airbus A320-100/200...,2230,"[EVN, TBS, AER, SVO]",1,1055,"[3F, WZ, DP]","[1280, 950]","[[1746646200, 1746726600], [1747215300, 174726...","[[DME, TBS], [TBS, SVO]]",5.881333,"[has_night_transfer, night_transfer, long_layo...",630,rub,36228,6500241,"[[False, False], [False, False]]","[[1PCx30x20x40, 1PCx30x20x40], [1PC5x30x20x40,...","[[{}, {}], [{'return_before_flight': {'availab...",1PC5x30x20x40,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,1280,950,"[{}, {}]",[{'return_before_flight': {'available': False}...,40000,False,False,19,9,False
688,[{'flight': [{'aircraft': 'Airbus A320-100/200...,1855,"[EVN, TBS, AER, VKO]",1,1055,"[3F, WZ, DP]","[1280, 575]","[[1746646200, 1746726600], [1747215300, 174724...","[[DME, TBS], [TBS, VKO]]",6.039667,"[night_transfer, long_layover, overnight_layov...",632,rub,36228,6500240,"[[False, False], [False, False]]","[[1PCx30x20x40, 1PCx30x20x40], [1PC5x30x20x40,...","[[{}, {}], [{'return_before_flight': {'availab...",1PC5x30x20x40,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,1280,575,"[{}, {}]",[{'return_before_flight': {'available': False}...,40000,False,False,19,9,False
689,[{'flight': [{'aircraft': 'Airbus A320-100/200...,1900,"[EVN, TBS, AER, SVO]",1,1055,"[WZ, DP, 3F]","[1280, 620]","[[1746646200, 1746726600], [1747215300, 174724...","[[DME, TBS], [TBS, SVO]]",6.030667,"[has_night_transfer, night_transfer, long_layo...",633,rub,36228,6500242,"[[False, False], [False, False]]","[[1PCx30x20x40, 1PCx30x20x40], [1PC5x30x20x40,...","[[{}, {}], [{'return_before_flight': {'availab...",1PC5x30x20x40,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,1280,620,"[{}, {}]",[{'return_before_flight': {'available': False}...,40000,False,False,19,9,False
739,[{'flight': [{'aircraft': 'Boeing 737-800 (win...,2995,"[EVN, TBS, AER, VKO]",1,1370,"[WZ, DP, SU, 3F]","[1335, 1660]","[[1746642900, 1746726600], [1747215300, 174731...","[[SVO, TBS], [TBS, VKO]]",5.569000,"[has_night_transfer, overnight_layover, night_...",683,rub,36598,6500292,"[[False, False], [False, False]]","[[1PC10x40x25x55, 1PCx30x20x40], [1PC5x30x20x4...",[[{'return_before_flight': {'available': False...,1PC5x30x20x40,,True,False,{'flight': [{'aircraft': 'Boeing 737-800 (wing...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,1335,1660,[{'return_before_flight': {'available': False}...,[{'return_before_flight': {'available': False}...,40000,False,False,18,9,False
748,[{'flight': [{'aircraft': 'Airbus A320-100/200...,1830,"[EVN, TBS, AER, DME]",1,1055,"[3F, WZ, U6]","[1280, 550]","[[1746646200, 1746726600], [1747215300, 174724...","[[DME, TBS], [TBS, DME]]",6.044667,"[has_night_transfer, long_layover, overnight_l...",692,rub,36763,6500302,"[[False, False], [False, False]]","[[1PCx30x20x40, 1PCx30x20x40], [1PC5x30x20x40,...","[[{}, {}], [{'return_before_flight': {'availab...",1PC5x30x20x40,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,1280,550,"[{}, {}]",[{'return_before_flight': {'available': False}...,40000,False,False,19,9,False


прочие (dummy):

In [16]:
viable_proposals_other = viable_proposals_other[
    ~viable_proposals_other.idx.isin(indices_already_displayed)
].copy()
viable_proposals_other.loc[
    viable_proposals_other.groupby("idx")["price"].idxmin()
].reset_index(drop=True).sort_values("total_duration").iloc[:5]

,segment,total_duration,stops_airports,max_stops,max_stop_duration,carriers,segment_durations,segments_time,segments_airports,segments_rating,tags,idx,currency,price,url,flights_baggage,flights_handbags,flight_additional_tariff_infos,viable_handbags,viable_baggage,has_handbags,has_baggage,flight_to,flight_back,duration_to,duration_back,tariff_to,tariff_back,rounded_price_5k,has_exchange,has_return,flight_to_time,flight_back_time,has_blacklisted_airlines
17,"[{'flight': [{'aircraft': 'Boeing 737-800', 'a...",340,"[TBS, ZIA]",0,0,"[WZ, A9]","[170, 170]","[[1746597600, 1746611400], [1747229400, 174723...","[[VKO, TBS], [TBS, ZIA]]",6.842444,"[convenient_ticket, direct]",1484,rub,57504,18300497,"[[1PC25], [1PC20]]","[[1PC8x40x25x55], [1PC5x30x20x40]]",[[{'return_before_flight': {'available': False...,1PC5x30x20x40,1PC20,True,True,"{'flight': [{'aircraft': 'Boeing 737-800', 'ar...",{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,170,170,[{'return_before_flight': {'available': False}...,[{'return_before_flight': {'available': False}...,60000,False,False,6,13,False
1,"[{'flight': [{'aircraft': 'Boeing 737-800', 'a...",355,"[TBS, VKO]",0,0,"[A9, A4]","[170, 185]","[[1746637200, 1746651000], [1747247700, 174725...","[[VKO, TBS], [TBS, VKO]]",8.087593,"[convenient_ticket, direct]",1301,rub,63715,6500015,"[[1PC25], [1PC23]]","[[1PC8x40x25x55], [1PC5x40x20x55]]","[[{}], [{'return_before_flight': {'available':...",1PC5x40x20x55,1PC23,True,True,"{'flight': [{'aircraft': 'Boeing 737-800', 'ar...",{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,170,185,[{}],[{'return_before_flight': {'available': False}...,60000,False,False,17,18,False
16,"[{'flight': [{'aircraft': 'Boeing 737-800', 'a...",360,"[TBS, VKO]",0,0,"[A9, A4]","[170, 190]","[[1746597600, 1746611400], [1747230600, 174723...","[[VKO, TBS], [TBS, VKO]]",6.860667,"[convenient_ticket, direct]",1483,rub,57410,18300496,"[[1PC25], [1PC23]]","[[1PC8x40x25x55], [1PC5x40x20x55]]",[[{'return_before_flight': {'available': False...,1PC5x40x20x55,1PC23,True,True,"{'flight': [{'aircraft': 'Boeing 737-800', 'ar...",{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,170,190,[{'return_before_flight': {'available': False}...,"[{'return_before_flight': {'available': True},...",60000,True,False,6,13,False
14,[{'flight': [{'aircraft': 'Sukhoi Superjet 100...,370,"[TBS, VKO]",0,0,"[WZ, A9]","[190, 180]","[[1746606000, 1746621000], [1747261800, 174726...","[[ZIA, TBS], [TBS, VKO]]",9.001111,"[convenient_ticket, direct]",1481,rub,57008,18300492,"[[1PC20], [1PC25]]","[[1PC5x30x20x40], [1PC8x40x25x55]]",[[{'return_before_flight': {'available': True}...,1PC5x30x20x40,1PC20,True,True,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,"{'flight': [{'aircraft': 'Boeing 737-800', 'ar...",190,180,"[{'return_before_flight': {'available': True},...",[{'return_before_flight': {'available': False}...,60000,True,False,8,22,False
15,[{'flight': [{'aircraft': 'Sukhoi Superjet 100...,370,"[TBS, VKO]",0,0,"[WZ, A9]","[190, 180]","[[1746606000, 1746621000], [1747231200, 174723...","[[ZIA, TBS], [TBS, VKO]]",8.267778,"[direct, convenient_ticket]",1482,rub,57008,18300494,"[[1PC20], [1PC25]]","[[1PC5x30x20x40], [1PC8x40x25x55]]",[[{'return_before_flight': {'available': True}...,1PC5x30x20x40,1PC20,True,True,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,"{'flight': [{'aircraft': 'Boeing 737-800', 'ar...",190,180,"[{'return_before_flight': {'available': True},...",[{'return_before_flight': {'available': False}...,60000,True,False,8,14,False


```Qatar Airways, прямой
    10 мая, Шереметьево SVO 17:05 – Хамад DOH 22:30
    🔹 110 476 руб. / за 3 пассажиров, багаж 25 кг / обмен, возврат со штрафом
    
    Air Arabia, с пересадкой в Шардже
    10 мая, Домодедово DME 00:10 – Шарджа SHJ 08:45
    Пересадка 2ч5мин
    10 мая, Шарджа SHJ 08:45 – Хамад DOH 08:50
    🔹 86 195 руб. / за 3 пассажиров, багаж 20 кг / обмен со штрафом, возврат в форме
    ваучера

In [17]:
smpl = viable_proposals_other.iloc[0]

In [26]:
template_res = ""

airlines = []

url_ = smpl.url

flight_info_to = smpl.flight_to["flight"]
transfers_info_to = smpl.flight_to.get("transfers", None)

flight_info_back = smpl.flight_back.get("flight", None)
transfers_info_back = smpl.flight_back.get("transfers", None)

class_ = flight_info_to[0]["trip_class"]
class_ = "Эконом, " if class_ == "Y" else ("Бизнес, " if class_ == "C" else "")

# TO ---

all_carriers = [airlines_mapping.loc[i["operating_carrier"]]["name"] for i in flight_info_to + flight_info_back]
all_carriers = sorted(set(all_carriers), key=lambda x: all_carriers.index(x))
template_res += "**" + ", ".join(all_carriers) + "**\n\n"

for c, (flight_info, transfers_info) in enumerate(zip(
    [flight_info_to, flight_info_back],
    [transfers_info_to, transfers_info_back]
)):
    if c == 0 and flight_info_back:
        template_res += "Туда - "
    if c == 1 and flight_info_back:
        template_res += "Обратно - "

    stops_all = []
    stops_all_info = []

    if transfers_info:
        for transfer_i in transfers_info:
            stops_all.append(transfer_i["at"])
            stops_all_info.append(transfer_i["duration_seconds"])

    if len(stops_all) == 1:
        template_res += f"Пересадка в городе {', '.join(cities_mapping.loc[airports_mapping.loc[stops_all]['city_code']]['name'])}:\n\n"
    elif len(stops_all) > 1:
        template_res += f"Пересадки в городах {', '.join(cities_mapping.loc[airports_mapping.loc[stops_all]['city_code']]['name'])}:\n\n"
    else:
        template_res += "Прямой:\n\n"

    for c, flight_i in enumerate(flight_info):
        if c > 0:
            template_res += f"Пересадка {format_seconds(stops_all_info[c-1])}\n\n"
        airport_to, airport_from = flight_i["departure"], flight_i["arrival"]
        airport_to_naming, airport_from_naming = airports_mapping.loc[airport_to]["name"], airports_mapping.loc[airport_from]["name"]
        template_res += f"{format_date_russian(flight_i['departure_date'])}, {airport_to_naming} {airport_to} - {flight_i['departure_time']} {airport_from_naming} {airport_from} {flight_i['arrival_time']}\n\n"

pass_string = ""
for k, v in AVIASALES_JSON.items():
    if k == "adults":
        pass_string += format_passengers(v, "adults")
    elif k == "children":
        if v > 0:
            pass_string += format_passengers(v, "children")
    elif k == "infants":
        if v > 0:
            pass_string += format_passengers(v, "infants")
        
handbag_string = f"ручная кладь {smpl.viable_handbags.replace('1PC', '')}кг" if smpl.viable_handbags else "без ручной клади"
baggage_string = f"багаж {smpl.viable_baggage.replace('1PC', '')}кг" if smpl.viable_baggage else "без багажа"

template_res += f"🔹 {smpl.price} руб. / за {pass_string}, {handbag_string}, {baggage_string} "\
                f"/ {class_}{summarize_exchange_return(smpl.tariff_to, smpl.tariff_back, 'get_str')} [{url_}]"

### (additional options)
options_list = ["has_handbags", "has_baggage", "has_exchange", "has_return"]

other_variants = (
    proposals_df_full[proposals_df_full.idx == smpl.idx]
    .sort_values("price")
    .drop_duplicates(subset=options_list, keep="first")
).iloc[:2]
possible_improvements = []

improvements_mapping = {
    "has_handbags": "с ручной кладью",
    "has_baggage": "с багажом",
    "has_exchange": "с обменом",
    "has_return": "с возвратом"
}

for i in options_list:
    if smpl[i] == False:
        possible_improvements.append(i)

if other_variants.shape[0] > 1:
    other_improvements = []
    for idx, row in other_variants.iterrows():
        row_improvements_i = {idx: []}
        for i in possible_improvements:
            if row[i] == True:
                row_improvements_i[idx].append(i)
        other_improvements.append(row_improvements_i)
    if len(row_improvements_i):
        template_res += "\n\nВозможные улучшения:"
        for row_improvements_i in other_improvements:
            for k, v in row_improvements_i.items():
                if len(v) > 0:
                    price_i = other_variants.loc[k, "price"]
                    url_i = other_variants.loc[k, "url"]
                    template_res += f"\n\n🔹 {price_i} руб. {', '.join([improvements_mapping[i] for i in v])} [{url_i}]"

template_res += "\n\n---\n\n"

print(template_res)

**Azal**

Туда - Пересадка в городе Баку:

7 мая, Внуково VKO - 01:00 Гейдар Алиев GYD 05:25

Пересадка 9ч 15мин

7 мая, Гейдар Алиев GYD - 14:40 Шота Руставели TBS 15:50

Обратно - Пересадка в городе Баку:

14 мая, Шота Руставели TBS - 02:40 Гейдар Алиев GYD 03:50

Пересадка 5ч 5мин

14 мая, Гейдар Алиев GYD - 08:55 Внуково VKO 11:15

🔹 65752 руб. / за 1 взрослого, ручная кладь 10x40x23x55кг, без багажа / Эконом, без обмена, без возврата [18300721]

Возможные улучшения:

🔹 76551 руб. с багажом, с обменом, с возвратом [18300722]

---




In [62]:
smpl

segment                           [{'flight': [{'aircraft': 'Airbus A320-100/200...
total_duration                                                                  610
stops_airports                                                      [GYD, TBS, VKO]
max_stops                                                                         1
max_stop_duration                                                               150
carriers                                                                   [J2, A4]
segment_durations                                                        [420, 190]
segments_time                     [[1746578400, 1746607200], [1747230600, 174723...
segments_airports                                          [[DME, TBS], [TBS, VKO]]
segments_rating                                                            7.260667
tags                              [has_night_transfer, night_transfer, overnight...
idx                                                                         

In [63]:
smpl

segment                           [{'flight': [{'aircraft': 'Airbus A320-100/200...
total_duration                                                                  610
stops_airports                                                      [GYD, TBS, VKO]
max_stops                                                                         1
max_stop_duration                                                               150
carriers                                                                   [J2, A4]
segment_durations                                                        [420, 190]
segments_time                     [[1746578400, 1746607200], [1747230600, 174723...
segments_airports                                          [[DME, TBS], [TBS, VKO]]
segments_rating                                                            7.260667
tags                              [has_night_transfer, night_transfer, overnight...
idx                                                                         

In [61]:
summarize_exchange_return(smpl.tariff_to, smpl.tariff_back, 'get_str')

'с обменом, с возвратом'

In [54]:
other_variants

,segment,total_duration,stops_airports,max_stops,max_stop_duration,carriers,segment_durations,segments_time,segments_airports,segments_rating,tags,idx,currency,price,url,flights_baggage,flights_handbags,flight_additional_tariff_infos,viable_handbags,viable_baggage,has_handbags,has_baggage,flight_to,flight_back,duration_to,duration_back,tariff_to,tariff_back,rounded_price_5k,has_exchange,has_return,flight_to_time,flight_back_time,has_blacklisted_airlines
718,[{'flight': [{'aircraft': 'Airbus A320-100/200...,610,"[GYD, TBS, VKO]",1,150,"[J2, A4]","[420, 190]","[[1746578400, 1746607200], [1747230600, 174723...","[[DME, TBS], [TBS, VKO]]",7.260667,"[has_night_transfer, night_transfer, overnight...",298,rub,64212,18300587,"[[1PC23, 1PC23], [1PC23]]","[[1PC10x40x23x55, 1PC10x40x23x55], [1PC5x40x20...",[[{'return_before_flight': {'available': True}...,1PC5x40x20x55,1PC23,True,True,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,420,190,"[{'return_before_flight': {'available': True},...","[{'return_before_flight': {'available': True},...",60000,True,True,0,13,True
719,[{'flight': [{'aircraft': 'Airbus A320-100/200...,610,"[GYD, TBS, VKO]",1,150,"[J2, A4]","[420, 190]","[[1746578400, 1746607200], [1747230600, 174723...","[[DME, TBS], [TBS, VKO]]",7.260667,"[has_night_transfer, night_transfer, overnight...",298,rub,59554,18300586,"[[False, False], [1PC23]]","[[1PC10x40x23x55, 1PC10x40x23x55], [1PC5x40x20...",[[{'return_before_flight': {'available': False...,1PC5x40x20x55,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,420,190,[{'return_before_flight': {'available': False}...,"[{'return_before_flight': {'available': True},...",60000,False,False,0,13,True


In [53]:
possible_improvements

['has_baggage', 'has_exchange', 'has_return']

In [46]:
smpl["has_handbags"]

True

In [39]:
other_variants = proposals_df_full[proposals_df_full.idx == smpl.idx]

In [44]:
other_variants.drop(718)

,segment,total_duration,stops_airports,max_stops,max_stop_duration,carriers,segment_durations,segments_time,segments_airports,segments_rating,tags,idx,currency,price,url,flights_baggage,flights_handbags,flight_additional_tariff_infos,viable_handbags,viable_baggage,has_handbags,has_baggage,flight_to,flight_back,duration_to,duration_back,tariff_to,tariff_back,rounded_price_5k,has_exchange,has_return,flight_to_time,flight_back_time,has_blacklisted_airlines
719,[{'flight': [{'aircraft': 'Airbus A320-100/200...,610,"[GYD, TBS, VKO]",1,150,"[J2, A4]","[420, 190]","[[1746578400, 1746607200], [1747230600, 174723...","[[DME, TBS], [TBS, VKO]]",7.260667,"[has_night_transfer, night_transfer, overnight...",298,rub,59554,18300586,"[[False, False], [1PC23]]","[[1PC10x40x23x55, 1PC10x40x23x55], [1PC5x40x20...",[[{'return_before_flight': {'available': False...,1PC5x40x20x55,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,420,190,[{'return_before_flight': {'available': False}...,"[{'return_before_flight': {'available': True},...",60000,False,False,0,13,True


In [38]:
other_variants

NameError: name 'other_variants' is not defined

In [37]:
proposals_df_full[(proposals_df_full.idx == smpl.idx)]

,segment,total_duration,stops_airports,max_stops,max_stop_duration,carriers,segment_durations,segments_time,segments_airports,segments_rating,tags,idx,currency,price,url,flights_baggage,flights_handbags,flight_additional_tariff_infos,viable_handbags,viable_baggage,has_handbags,has_baggage,flight_to,flight_back,duration_to,duration_back,tariff_to,tariff_back,rounded_price_5k,has_exchange,has_return,flight_to_time,flight_back_time,has_blacklisted_airlines
718,[{'flight': [{'aircraft': 'Airbus A320-100/200...,610,"[GYD, TBS, VKO]",1,150,"[J2, A4]","[420, 190]","[[1746578400, 1746607200], [1747230600, 174723...","[[DME, TBS], [TBS, VKO]]",7.260667,"[has_night_transfer, night_transfer, overnight...",298,rub,64212,18300587,"[[1PC23, 1PC23], [1PC23]]","[[1PC10x40x23x55, 1PC10x40x23x55], [1PC5x40x20...",[[{'return_before_flight': {'available': True}...,1PC5x40x20x55,1PC23,True,True,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,420,190,"[{'return_before_flight': {'available': True},...","[{'return_before_flight': {'available': True},...",60000,True,True,0,13,True
719,[{'flight': [{'aircraft': 'Airbus A320-100/200...,610,"[GYD, TBS, VKO]",1,150,"[J2, A4]","[420, 190]","[[1746578400, 1746607200], [1747230600, 174723...","[[DME, TBS], [TBS, VKO]]",7.260667,"[has_night_transfer, night_transfer, overnight...",298,rub,59554,18300586,"[[False, False], [1PC23]]","[[1PC10x40x23x55, 1PC10x40x23x55], [1PC5x40x20...",[[{'return_before_flight': {'available': False...,1PC5x40x20x55,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,420,190,[{'return_before_flight': {'available': False}...,"[{'return_before_flight': {'available': True},...",60000,False,False,0,13,True


In [34]:
smpl

segment                           [{'flight': [{'aircraft': 'Airbus A320-100/200...
total_duration                                                                  610
stops_airports                                                      [GYD, TBS, VKO]
max_stops                                                                         1
max_stop_duration                                                               150
carriers                                                                   [J2, A4]
segment_durations                                                        [420, 190]
segments_time                     [[1746578400, 1746607200], [1747230600, 174723...
segments_airports                                          [[DME, TBS], [TBS, VKO]]
segments_rating                                                            7.260667
tags                              [has_night_transfer, night_transfer, overnight...
idx                                                                         

In [31]:
proposals_df_full[proposals_df_full.idx == smpl.idx]

,segment,total_duration,stops_airports,max_stops,max_stop_duration,carriers,segment_durations,segments_time,segments_airports,segments_rating,tags,idx,currency,price,url,flights_baggage,flights_handbags,flight_additional_tariff_infos,viable_handbags,viable_baggage,has_handbags,has_baggage,flight_to,flight_back,duration_to,duration_back,tariff_to,tariff_back,rounded_price_5k,has_exchange,has_return,flight_to_time,flight_back_time,has_blacklisted_airlines
718,[{'flight': [{'aircraft': 'Airbus A320-100/200...,610,"[GYD, TBS, VKO]",1,150,"[J2, A4]","[420, 190]","[[1746578400, 1746607200], [1747230600, 174723...","[[DME, TBS], [TBS, VKO]]",7.260667,"[has_night_transfer, night_transfer, overnight...",298,rub,64212,18300587,"[[1PC23, 1PC23], [1PC23]]","[[1PC10x40x23x55, 1PC10x40x23x55], [1PC5x40x20...",[[{'return_before_flight': {'available': True}...,1PC5x40x20x55,1PC23,True,True,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,420,190,"[{'return_before_flight': {'available': True},...","[{'return_before_flight': {'available': True},...",60000,True,True,0,13,True
719,[{'flight': [{'aircraft': 'Airbus A320-100/200...,610,"[GYD, TBS, VKO]",1,150,"[J2, A4]","[420, 190]","[[1746578400, 1746607200], [1747230600, 174723...","[[DME, TBS], [TBS, VKO]]",7.260667,"[has_night_transfer, night_transfer, overnight...",298,rub,59554,18300586,"[[False, False], [1PC23]]","[[1PC10x40x23x55, 1PC10x40x23x55], [1PC5x40x20...",[[{'return_before_flight': {'available': False...,1PC5x40x20x55,,True,False,{'flight': [{'aircraft': 'Airbus A320-100/200'...,{'flight': [{'aircraft': 'Sukhoi Superjet 100'...,420,190,[{'return_before_flight': {'available': False}...,"[{'return_before_flight': {'available': True},...",60000,False,False,0,13,True


In [30]:
smpl.idx

298

In [24]:
smpl.flight_back

{'flight': [{'aircraft': 'Sukhoi Superjet 100',
   'arrival': 'VKO',
   'arrival_date': '2025-05-14',
   'arrival_time': '16:00',
   'arrival_timestamp': 1747227600,
   'delay': 0,
   'departure': 'TBS',
   'departure_date': '2025-05-14',
   'departure_time': '13:50',
   'departure_timestamp': 1747216200,
   'duration': 190,
   'equipment': 'SU9',
   'local_arrival_timestamp': 1747238400,
   'local_departure_timestamp': 1747230600,
   'marketing_carrier': 'A4',
   'number': '7010',
   'operating_carrier': 'A4',
   'operated_by': 'A4',
   'rating': 0,
   'technical_stops': None,
   'trip_class': 'Y'}],
 'rating': {'total': 7.785999999999999}}

In [19]:
transfers_info

# TODO

- URL FORMATION
- RANKING (check notes)
- TAGS

In [18]:
smpl

segment                           [{'flight': [{'aircraft': 'Airbus A321-100/200...
total_duration                                                                 1720
stops_airports                                            [SAW, TBS, SAW, ADB, VKO]
max_stops                                                                         2
max_stop_duration                                                               570
carriers                                                                       [PC]
segment_durations                                                       [605, 1115]
segments_time                     [[1746630300, 1746670200], [1747206900, 174727...
segments_airports                                          [[VKO, TBS], [TBS, VKO]]
segments_rating                                                            5.599667
tags                                                                 [long_layover]
idx                                                                         

In [26]:
smpl.tariff_back

[{'return_before_flight': {'available': True},
  'change_before_flight': {'available': True}},
 {'return_before_flight': {'available': True},
  'return_after_flight': {'available': False},
  'change_before_flight': {'available': True},
  'change_after_flight': {'available': False}},
 {'return_before_flight': {'available': True},
  'return_after_flight': {'available': False},
  'change_before_flight': {'available': True},
  'change_after_flight': {'available': False}}]

In [9]:
flight_i

{'aircraft': 'Airbus A321-100/200',
 'arrival': 'TBS',
 'arrival_date': '2025-05-08',
 'arrival_time': '02:10',
 'arrival_timestamp': 1746655800,
 'delay': 215,
 'departure': 'SAW',
 'departure_date': '2025-05-07',
 'departure_time': '22:55',
 'departure_timestamp': 1746647700,
 'duration': 135,
 'equipment': '321',
 'local_arrival_timestamp': 1746670200,
 'local_departure_timestamp': 1746658500,
 'marketing_carrier': 'PC',
 'number': '318',
 'operating_carrier': 'PC',
 'operated_by': 'PC',
 'rating': 0,
 'technical_stops': None,
 'trip_class': 'Y'}

In [32]:
smpl.flight_back

{'flight': [{'aircraft': 'Canadair CRJ 200',
   'arrival': 'NOJ',
   'arrival_date': '2025-05-14',
   'arrival_time': '09:30',
   'arrival_timestamp': 1747197000,
   'delay': 0,
   'departure': 'UUA',
   'departure_date': '2025-05-14',
   'departure_time': '05:00',
   'departure_timestamp': 1747188000,
   'duration': 150,
   'equipment': 'CR2',
   'local_arrival_timestamp': 1747215000,
   'local_departure_timestamp': 1747198800,
   'marketing_carrier': 'RT',
   'number': '337',
   'operating_carrier': 'RT',
   'operated_by': 'RT',
   'rating': 0,
   'technical_stops': None,
   'trip_class': 'Y',
   'seats': 4},
  {'aircraft': 'Embraer EMB 170 / EMB 175',
   'arrival': 'OVB',
   'arrival_date': '2025-05-15',
   'arrival_time': '12:35',
   'arrival_timestamp': 1747287300,
   'delay': 1405,
   'departure': 'NOJ',
   'departure_date': '2025-05-15',
   'departure_time': '08:55',
   'departure_timestamp': 1747281300,
   'duration': 100,
   'equipment': 'E70',
   'local_arrival_timestamp': 17

In [8]:
smpl.flight_to["flight"]

[{'aircraft': 'Airbus A319',
  'arrival': 'LED',
  'arrival_date': '2025-05-07',
  'arrival_time': '06:15',
  'arrival_timestamp': 1746587700,
  'delay': 0,
  'departure': 'SGC',
  'departure_date': '2025-05-07',
  'departure_time': '04:55',
  'departure_timestamp': 1746575700,
  'duration': 200,
  'equipment': '319',
  'local_arrival_timestamp': 1746598500,
  'local_departure_timestamp': 1746593700,
  'marketing_carrier': 'SU',
  'number': '6442',
  'operating_carrier': 'FV',
  'operated_by': 'FV',
  'rating': 0,
  'technical_stops': None,
  'trip_class': 'Y',
  'seats': 4},
 {'aircraft': 'Sukhoi Superjet 100',
  'arrival': 'VKO',
  'arrival_date': '2025-05-08',
  'arrival_time': '20:30',
  'arrival_timestamp': 1746725400,
  'delay': 2205,
  'departure': 'LED',
  'departure_date': '2025-05-08',
  'departure_time': '19:00',
  'departure_timestamp': 1746720000,
  'duration': 90,
  'equipment': 'SU9',
  'local_arrival_timestamp': 1746736200,
  'local_departure_timestamp': 1746730800,
  '

In [6]:
proposals_df_full.viable_handbags.value_counts()

viable_handbags
5x30x15x40    3202
5x35x15x45     394
5x27x15x36     342
Name: count, dtype: int64

In [10]:
proposals_df_full.iloc[0].segment[]

{'flight': [{'aircraft': 'Antonow / Antonov An-24',
   'arrival': 'HMA',
   'arrival_date': '2025-05-07',
   'arrival_time': '07:40',
   'arrival_timestamp': 1746585600,
   'delay': 0,
   'departure': 'SGC',
   'departure_date': '2025-05-07',
   'departure_time': '06:35',
   'departure_timestamp': 1746581700,
   'duration': 65,
   'equipment': 'AN4',
   'local_arrival_timestamp': 1746603600,
   'local_departure_timestamp': 1746599700,
   'marketing_carrier': 'UT',
   'number': '115',
   'operating_carrier': 'UT',
   'operated_by': 'UT',
   'rating': 0,
   'technical_stops': None,
   'trip_class': 'Y',
   'seats': 2},
  {'aircraft': 'Boeing 737-500 (winglets)',
   'arrival': 'VKO',
   'arrival_date': '2025-05-08',
   'arrival_time': '09:00',
   'arrival_timestamp': 1746684000,
   'delay': 1435,
   'departure': 'HMA',
   'departure_date': '2025-05-08',
   'departure_time': '07:35',
   'departure_timestamp': 1746671700,
   'duration': 205,
   'equipment': '73E',
   'local_arrival_timestam

In [112]:
proposals_df_full.viable_handbags.sort_values(ascending=False)

3937    1PC5x35x15x45
659     1PC5x35x15x45
1551    1PC5x35x15x45
1550    1PC5x35x15x45
655     1PC5x35x15x45
            ...      
882     1PC5x27x15x36
881     1PC5x27x15x36
880     1PC5x27x15x36
879     1PC5x27x15x36
552     1PC5x27x15x36
Name: viable_handbags, Length: 3938, dtype: object

In [108]:
proposals_df_full.viable_baggage.sort_values(ascending=False)

1969    1PC10
1641    1PC10
1633    1PC10
1635    1PC10
3128    1PC10
        ...  
1752         
1754         
1756         
1759         
3937         
Name: viable_baggage, Length: 3938, dtype: object

In [117]:
proposals_df_full

,segment,total_duration,stops_airports,max_stops,max_stop_duration,carriers,segment_durations,segments_time,segments_airports,segments_rating,tags,idx,currency,price,url,flights_baggage,flights_handbags,flight_additional_tariff_infos,viable_baggage,viable_handbags
0,[{'flight': [{'aircraft': 'Antonow / Antonov A...,6030,"[HMA, VKO, UUA, NOJ, OVB, SGC]",2,1595,"[UT, RT, S7]","[2630, 3400]","[[1746599700, 1746750300], [1747198800, 174741...","[[SGC, UUA], [UUA, SGC]]",3.114000,"[has_night_transfer, long_layover, overnight_l...",0,rub,58761,7000046,"[[False, False, 1PC20], [1PC10, False, False]]","[[1PC5x30x20x40, 1PC5x30x20x40, 1PC5x35x15x45]...",[[{'return_before_flight': {'available': False...,,1PC5x30x15x40
1,[{'flight': [{'aircraft': 'Antonow / Antonov A...,6030,"[HMA, VKO, UUA, NOJ, OVB, SGC]",2,1595,"[UT, RT, S7]","[2630, 3400]","[[1746599700, 1746750300], [1747198800, 174741...","[[SGC, UUA], [UUA, SGC]]",3.114000,"[has_night_transfer, long_layover, overnight_l...",0,rub,66444,7000047,"[[1PC20, 1PC20, 1PC20], [1PC10, 1PC23, 1PC20]]","[[1PC5x30x20x40, 1PC5x30x20x40, 1PC5x35x15x45]...",[[{'return_before_flight': {'available': False...,1PC10,1PC5x30x15x40
2,[{'flight': [{'aircraft': 'Sukhoi Superjet 100...,5770,"[SVX, SVO, UUA, NOJ, TJM, SGC]",2,3230,"[WZ, 5N, RT, YC, UT]","[1940, 3830]","[[1746641100, 1746750300], [1747198800, 174743...","[[SGC, UUA], [UUA, SGC]]",3.246667,"[night_transfer, has_night_transfer, airport_c...",1,rub,60890,7000026,"[[False, False, 1PC20], [1PC10, 1PC23, False]]","[[1PC5x30x20x40, 1PC10x30x20x40, 1PC5x35x15x45...",[[{'return_before_flight': {'available': False...,,1PC5x30x15x40
3,[{'flight': [{'aircraft': 'Sukhoi Superjet 100...,5770,"[SVX, SVO, UUA, NOJ, TJM, SGC]",2,3230,"[WZ, 5N, RT, YC, UT]","[1940, 3830]","[[1746641100, 1746750300], [1747198800, 174743...","[[SGC, UUA], [UUA, SGC]]",3.246667,"[night_transfer, has_night_transfer, airport_c...",1,rub,69261,7000027,"[[1PC20, 1PC10, 1PC20], [1PC10, 1PC23, 1PC20]]","[[1PC5x30x20x40, 1PC10x30x20x40, 1PC5x35x15x45...",[[{'return_before_flight': {'available': True}...,1PC10,1PC5x30x15x40
4,[{'flight': [{'aircraft': 'Sukhoi Superjet 100...,5340,"[SVX, SVO, UUA, NOJ, OVB, SGC]",2,1595,"[WZ, 5N, RT, S7, UT]","[1940, 3400]","[[1746641100, 1746750300], [1747198800, 174741...","[[SGC, UUA], [UUA, SGC]]",3.418667,"[airport_change, has_night_transfer, long_layo...",2,rub,61013,7000072,"[[False, False, 1PC20], [1PC10, False, False]]","[[1PC5x30x20x40, 1PC10x30x20x40, 1PC5x35x15x45...",[[{'return_before_flight': {'available': False...,,1PC5x30x15x40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3933,[{'flight': [{'aircraft': 'Embraer EMB 170 / E...,3575,"[OVB, DME, UUA, NOJ, SLY, LED, SGC]",3,1280,"[S7, RT, YC, FV]","[2330, 1245]","[[1746617700, 1746750300], [1747198800, 174728...","[[SGC, UUA], [UUA, SGC]]",4.152333,"[night_transfer, has_night_transfer, airport_c...",1935,rub,121175,7003886,"[[1PC23, 1PC23, 1PC20], [1PC10, 1PC23, 1PC23, ...","[[1PC10x40x23x55, 1PC10x40x23x55, 1PC5x35x15x4...",[[{'return_before_flight': {'available': True}...,1PC10,1PC5x35x15x45
3934,[{'flight': [{'aircraft': 'Boeing 737-500 (win...,3395,"[GSV, SVO, UUA, NOJ, SLY, SVX, SVO, SGC]",4,915,"[UT, FV, RT, YC, SU]","[1965, 1430]","[[1746639600, 1746750300], [1747198800, 174729...","[[SGC, UUA], [UUA, SGC]]",4.201667,"[long_layover, overnight_layover, night_transf...",1936,rub,109586,7003879,"[[1PC20, False, 1PC20], [1PC10, 1PC23, 1PC23, ...","[[1PC10x40x25x55, 1PC10x40x25x55, 1PC5x35x15x4...",[[{'return_before_flight': {'available': True}...,,1PC5x35x15x45
3935,[{'flight': [{'aircraft': 'Boeing 737-500 (win...,3395,"[GSV, SVO, UUA, NOJ, SLY, SVX, SVO, SGC]",4,915,"[UT, FV, RT, YC, SU]","[1965, 1430]","[[1746639600, 1746750300], [1747198800, 174729...","[[SGC, UUA], [UUA, SGC]]",4.201667,"[long_layover, overnight_layover, night_transf...",1936,rub,113542,7003880,"[[1PC20, 1PC23, 1PC20], [1PC10, 1PC23, 1PC23, ...","[[1PC10x40x25x55, 1PC10x40x25x5

In [12]:
list(chain(*[[], [1,2,3]]))

[1, 2, 3]

In [11]:
proposals_df.iloc[0].segment

[{'flight': [{'aircraft': 'Airbus A320-100/200',
    'arrival': 'GYD',
    'arrival_date': '2025-05-07',
    'arrival_time': '16:30',
    'arrival_timestamp': 1746621000,
    'delay': 0,
    'departure': 'DME',
    'departure_date': '2025-05-07',
    'departure_time': '12:10',
    'departure_timestamp': 1746609000,
    'duration': 200,
    'equipment': '320',
    'local_arrival_timestamp': 1746635400,
    'local_departure_timestamp': 1746619800,
    'marketing_carrier': 'J2',
    'number': '182',
    'operating_carrier': 'J2',
    'operated_by': 'J2',
    'rating': 0,
    'technical_stops': None,
    'trip_class': 'Y',
    'seats': 6}],
  'rating': {'total': 8.638333333333334}}]

In [7]:
proposals_df.popularity.value_counts()

popularity
0    273
Name: count, dtype: int64

In [11]:
flight_info_list = [proposals[i]['segment'][0]['flight'] for i in range(len(proposals))]
flight_info_list_clean = [
    [
        {
            k: v
            for k, v in flight.items()
            if k in ["departure_date", "departure_time", "arrival_date", "arrival_time", "technical_stops"]
        }
        for flight in segment_flights
    ]
    for segment_flights in flight_info_list
]

In [12]:
proposals_df["precise_timings"] = flight_info_list_clean

In [13]:
proposals_df

,terms,segment,total_duration,stops_airports,is_charter,max_stops,max_stop_duration,carriers,segment_durations,segments_time,segments_airports,is_direct,popularity,segments_rating,tags,min_stop_duration,price,url,flights_baggage,flights_handbags,returns,precise_timings
0,"{'currency': 'rub', 'price': 24125, 'unified_p...",[{'flight': [{'aircraft': 'Airbus A320-100/200...,200,[GYD],False,0,0,[J2],[200],"[[1746619800, 1746635400]]","[[DME, GYD]]",True,0,8.638333,"[convenient_ticket, direct]",NaN,24125,38600003,[[]],[[1PC10x40x23x55]],[[{'return_before_flight': {'available': False...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."
1,"{'currency': 'rub', 'price': 26460, 'unified_p...",[{'flight': [{'aircraft': 'Airbus A320-100/200...,200,[GYD],False,0,0,[J2],[200],"[[1746578400, 1746594000]]","[[DME, GYD]]",True,0,5.463333,"[convenient_ticket, direct]",NaN,26460,38600000,[[]],[[1PC10x40x23x55]],[[{'return_before_flight': {'available': False...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."
2,"{'currency': 'rub', 'price': 29560, 'unified_p...",[{'flight': [{'aircraft': 'Airbus A320-100/200...,205,[GYD],False,0,0,[J2],[205],"[[1746579600, 1746595500]]","[[VKO, GYD]]",True,0,5.647833,"[convenient_ticket, direct]",NaN,29560,38600006,[[]],[[1PC10x40x23x55]],[[{'return_before_flight': {'available': False...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."
3,"{'currency': 'rub', 'price': 29560, 'unified_p...","[{'flight': [{'aircraft': 'Airbus A319', 'arri...",205,[GYD],False,0,0,[J2],[205],"[[1746622200, 1746638100]]","[[VKO, GYD]]",True,0,8.618667,"[convenient_ticket, direct]",NaN,29560,38600009,[[]],[[1PC10x40x23x55]],[[{'return_before_flight': {'available': False...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."
4,"{'currency': 'rub', 'price': 24978, 'unified_p...","[{'flight': [{'aircraft': 'Boeing 737-800', 'a...",195,[GYD],False,0,0,[UT],[195],"[[1746611400, 1746626700]]","[[VKO, GYD]]",True,0,9.066333,"[convenient_ticket, direct]",NaN,24978,25000000,[[1PC20]],[[1PC5x30x20x40]],[[{'return_before_flight': {'available': False...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,"{'currency': 'rub', 'price': 65394, 'unified_p...","[{'flight': [{'aircraft': 'Airbus A330-300', '...",510,"[IST, GYD]",False,1,90,[TK],[510],"[[1746599100, 1746633300]]","[[VKO, GYD]]",False,0,6.944000,[convenient_ticket],90.0,65394,6500077,"[[1PC30, 1PC30]]","[[1PC8x40x23x55, 1PC8x40x23x55]]",[[{'return_before_flight': {'available': True}...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."
269,"{'currency': 'rub', 'price': 65394, 'unified_p...","[{'flight': [{'aircraft': 'Airbus A330-300', '...",545,"[IST, GYD]",False,1,120,[TK],[545],"[[1746578100, 1746614400]]","[[VKO, GYD]]",False,0,6.923000,"[night_transfer, has_night_transfer]",120.0,65394,6500078,"[[1PC30, 1PC30]]","[[1PC8x40x23x55, 1PC8x40x23x55]]",[[{'return_before_flight': {'available': True}...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."
270,"{'currency': 'rub', 'price': 78015, 'unified_p...","[{'flight': [{'aircraft': 'Airbus A320neo', 'a...",1725,"[BAH, GYD]",False,1,1115,[GF],[1725],"[[1746607800, 1746714900]]","[[DME, GYD]]",False,0,5.465000,"[night_transfer, long_layover, has_night_trans...",1115.0,78015,6500081,"[[1PC25, 1PC25]]","[[1PC6x40x30x45, 1PC6x40x30x45]]","[[{}, {}]]","[{'arrival_date': '2025-05-07', 'arrival_time'..."
271,"{'currency': 'rub', 'price': 205112, 'unified_...","[{'flight': [{'aircraft': 'Airbus A380-800', '...",960,"[DXB, GYD]",False,1,460,"[EK, J2]",[960],"[[1746659400, 1746720600]]","[[DME, GYD]]",False,0,6.174000,"[convenient_ticket, long_layover]",460.0,205112,6500082,"[[1PC35, 1PC35]]","[[1PC7x38x20x55, 1PC10x40x23x55]]","[[{}, {'return_before_flight': {'available': F...","[{'arrival_date': '2025-05-08', 'arrival_time'..."


In [54]:
proposals_df.iloc[0].precise_timings

[{'arrival_date': '2025-05-07',
  'arrival_time': '07:40',
  'departure_date': '2025-05-07',
  'departure_time': '06:35',
  'technical_stops': None},
 {'arrival_date': '2025-05-08',
  'arrival_time': '09:00',
  'departure_date': '2025-05-08',
  'departure_time': '07:35',
  'technical_stops': None},
 {'arrival_date': '2025-05-09',
  'arrival_time': '00:25',
  'departure_date': '2025-05-08',
  'departure_time': '22:35',
  'technical_stops': None}]

In [53]:
proposals_df.head(5)

,terms,segment,total_duration,stops_airports,is_charter,max_stops,max_stop_duration,min_stop_duration,carriers,segment_durations,segments_time,segments_airports,is_direct,popularity,segments_rating,tags,price,url,flights_baggage,flights_handbags,returns,precise_timings
0,"{'currency': 'rub', 'price': 29342, 'unified_p...",[{'flight': [{'aircraft': 'Antonow / Antonov A...,2630,"[HMA, VKO, UUA]",False,2,1435,815,"[UT, RT]",[2630],"[[1746599700, 1746750300]]","[[SGC, UUA]]",False,0,3.422,"[has_night_transfer, overnight_layover, night_...",29342,7000010,"[[False, False, 1PC20]]","[[1PC5x30x20x40, 1PC5x30x20x40, 1PC5x35x15x45]]",[[{'return_before_flight': {'available': False...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."
1,"{'currency': 'rub', 'price': 31381, 'unified_p...",[{'flight': [{'aircraft': 'Boeing 737-800 (win...,1945,"[VKO, UUA]",False,1,1620,1620,"[UT, RT]",[1945],"[[1746640800, 1746750300]]","[[SGC, UUA]]",False,0,5.333,"[night_transfer, overnight_layover, long_layov...",31381,7000006,"[[False, 1PC20]]","[[1PC5x30x20x40, 1PC5x35x15x45]]",[[{'return_before_flight': {'available': False...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."
2,"{'currency': 'rub', 'price': 31580, 'unified_p...",[{'flight': [{'aircraft': 'Sukhoi Superjet 100...,1940,"[SVX, SVO, UUA]",False,2,895,675,"[RT, WZ, 5N]",[1940],"[[1746641100, 1746750300]]","[[SGC, UUA]]",False,0,4.336,"[has_night_transfer, overnight_layover, night_...",31580,7000014,"[[False, False, 1PC20]]","[[1PC5x30x20x40, 1PC10x30x20x40, 1PC5x35x15x45]]",[[{'return_before_flight': {'available': False...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."
3,"{'currency': 'rub', 'price': 31961, 'unified_p...",[{'flight': [{'aircraft': 'Sukhoi Superjet 100...,1940,"[SVX, DME, UUA]",False,2,965,610,"[WZ, U6, RT]",[1940],"[[1746641100, 1746750300]]","[[SGC, UUA]]",False,0,4.336,"[long_layover, has_night_transfer, airport_cha...",31961,7000019,"[[False, False, 1PC20]]","[[1PC5x30x20x40, 1PC10x40x20x55, 1PC5x35x15x45]]",[[{'return_before_flight': {'available': False...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."
4,"{'currency': 'rub', 'price': 31961, 'unified_p...",[{'flight': [{'aircraft': 'Sukhoi Superjet 100...,1940,"[SVX, DME, UUA]",False,2,920,655,"[WZ, U6, RT]",[1940],"[[1746641100, 1746750300]]","[[SGC, UUA]]",False,0,4.336,"[has_night_transfer, overnight_layover, night_...",31961,7000017,"[[False, False, 1PC20]]","[[1PC5x30x20x40, 1PC10x40x20x55, 1PC5x35x15x45]]",[[{'return_before_flight': {'available': False...,"[{'arrival_date': '2025-05-07', 'arrival_time'..."


---

LOGIC FOR TF-IDF BASED MATCHING:

In [12]:
with open("digital_assistant_first/utils/iata_airlines.json", "r") as f:
    iata_mapping = json.load(f)

In [19]:
iata_mapping_general = {i["name"]: i["code"] for i in iata_mapping if i["name"]}
iata_mapping_additional = {i["name_translations"]["en"]: i["code"] for i in iata_mapping
                           if i["name_translations"]["en"]}

In [20]:
iata_mapping_conc = {**iata_mapping_general, **iata_mapping_additional}

In [33]:
transformer = TfidfVectorizer(analyzer="char", ngram_range=(1, 2))
transformer.fit(list(iata_mapping_conc.keys()))

TfidfVectorizer(analyzer='char', ngram_range=(1, 2))

In [35]:
joblib.dump(transformer, "providers_iata_matcher_tfidf.joblib")

['providers_iata_matcher_tfidf.joblib']

In [39]:
providers_iata_matcher_vectors = transformer.transform(list(iata_mapping_conc.keys())).toarray()

In [57]:
matcher_dict = {i: providers_iata_matcher_vectors[c] for c, i in enumerate(list(iata_mapping_conc.values()))}

joblib.dump(
    matcher_dict,
    "providers_iata_matcher_vectors.joblib",
)

['providers_iata_matcher_vectors.joblib']

In [34]:
len(transformer.vocabulary_)

985

In [16]:
proposals_df.returns.iloc[0]

[[{'return_before_flight': {'available': False},
   'return_after_flight': {'available': False},
   'change_before_flight': {'available': True},
   'change_after_flight': {'available': True}},
  {'return_before_flight': {'available': False},
   'return_after_flight': {'available': False},
   'change_before_flight': {'available': True},
   'change_after_flight': {'available': True}},
  {'return_before_flight': {'available': True},
   'change_before_flight': {'available': True}}],
 [{'return_before_flight': {'available': True},
   'change_before_flight': {'available': True}},
  {'return_before_flight': {'available': False},
   'return_after_flight': {'available': False},
   'change_before_flight': {'available': True},
   'change_after_flight': {'available': False}},
  {'return_before_flight': {'available': False},
   'return_after_flight': {'available': False},
   'change_before_flight': {'available': True},
   'change_after_flight': {'available': True}}]]

{'70': {'VFLEXSNG,NNOR,BOW;PSOW,YBD3,YBDST,KNOR,KNOR': {'currency': 'rub',
   'price': 108281,
   'unified_price': 108281,
   'url': 7003888,
   'transfer_terms': [[{'is_virtual_interline': True},
     {'is_virtual_interline': True}],
    [{'is_virtual_interline': True},
     {'is_virtual_interline': False},
     {'is_virtual_interline': True},
     {'is_virtual_interline': False}]],
   'flight_additional_tariff_infos': [[{'return_before_flight': {'available': True},
      'return_after_flight': {'available': False},
      'change_before_flight': {'available': True},
      'change_after_flight': {'available': True}},
     {'return_before_flight': {'available': False},
      'return_after_flight': {'available': False},
      'change_before_flight': {'available': True},
      'change_after_flight': {'available': False}},
     {'return_before_flight': {'available': True},
      'change_before_flight': {'available': True}}],
    [{'return_before_flight': {'available': True},
      'change_

In [17]:
proposals_df["price"] = 

Index(['terms', 'xterms', 'segment', 'total_duration', 'stops_airports',
       'is_charter', 'max_stops', 'max_stop_duration', 'min_stop_duration',
       'carriers', 'segment_durations', 'segments_time', 'segments_airports',
       'sign', 'is_direct', 'flight_weight', 'popularity', 'segments_rating',
       'tags', 'validating_carrier'],
      dtype='object')

In [14]:
proposals_df

,terms,xterms,segment,total_duration,stops_airports,is_charter,max_stops,max_stop_duration,min_stop_duration,carriers,segment_durations,segments_time,segments_airports,sign,is_direct,flight_weight,popularity,segments_rating,tags,validating_carrier
0,"{'70': {'currency': 'rub', 'price': 58761, 'un...","{'70': {'NLTROW,OLTOW,BOW;PSOW,YBSOWEM,HLTOW':...",[{'flight': [{'aircraft': 'Antonow / Antonov A...,6030,"[HMA, VKO, UUA, NOJ, OVB, SGC]",False,2,1595,815,"[UT, RT, S7]","[2630, 3400]","[[1746599700, 1746750300], [1747198800, 174741...","[[SGC, UUA], [UUA, SGC]]",e1f96b4b40dd91bf6808007ae3cb2363,False,0.035575,0,3.114000,"[overnight_layover, long_layover, has_night_tr...",UT
1,"{'70': {'currency': 'rub', 'price': 60134, 'un...","{'70': {'NLTROW,OLTOW,BOW;PSOW,YBSOWEM,XBSOW':...",[{'flight': [{'aircraft': 'Antonow / Antonov A...,5745,"[HMA, VKO, UUA, NOJ, OVB, SGC]",False,2,1435,815,"[S7, UT, RT]","[2630, 3115]","[[1746599700, 1746750300], [1747198800, 174739...","[[SGC, UUA], [UUA, SGC]]",08cb3828d30141a54ebc9ded2f3e05a7,False,0.035575,0,3.228000,"[has_night_transfer, night_transfer, long_layo...",UT
2,"{'70': {'currency': 'rub', 'price': 61013, 'un...","{'70': {'VLTOW,PLTOW,BOW;PSOW,YBSOWEM,HLTOW': ...",[{'flight': [{'aircraft': 'Sukhoi Superjet 100...,5340,"[SVX, SVO, UUA, NOJ, OVB, SGC]",False,2,1595,675,"[5N, RT, S7, UT, WZ]","[1940, 3400]","[[1746641100, 1746750300], [1747198800, 174741...","[[SGC, UUA], [UUA, SGC]]",32882d39e9ae3aaa94fbea70c4a32700,False,0.032666,0,3.418667,"[airport_change, has_night_transfer, night_tra...",5N
3,"{'70': {'currency': 'rub', 'price': 61391, 'un...","{'70': {'NLTROW,NNOR,BOW;PSOW,YBSOWEM,HLTOW': ...",[{'flight': [{'aircraft': 'Antonow / Antonov A...,6030,"[HMA, SVO, UUA, NOJ, OVB, SGC]",False,2,1595,870,"[UT, SU, RT, S7]","[2630, 3400]","[[1746599700, 1746750300], [1747198800, 174741...","[[SGC, UUA], [UUA, SGC]]",eea3d82cce78b7ebde00da013cee2b1d,False,0.018526,0,3.114000,"[long_layover, overnight_layover, airport_chan...",SU
4,"{'70': {'currency': 'rub', 'price': 61574, 'un...","{'70': {'NSTROW,OSTDOW,BOW;PSOW,YSTOWEM,TSTOW'...",[{'flight': [{'aircraft': 'Antonow / Antonov A...,5155,"[HMA, VKO, UUA, NOJ, OVB, SGC]",False,2,1435,765,"[UT, RT, S7]","[2630, 2525]","[[1746599700, 1746750300], [1747198800, 174735...","[[SGC, UUA], [UUA, SGC]]",1f48b5c2ea4f1d674cbd12bd2c2332d1,False,0.035575,0,3.443000,"[long_layover, overnight_layover, has_night_tr...",UT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1929,"{'70': {'currency': 'rub', 'price': 106276, 'u...","{'70': {'TBSOW,TNOR,BOW;PSOW,YBD3,YBDFL,HNOR':...",[{'flight': [{'aircraft': 'Embraer EMB 170 / E...,3575,"[OVB, SVO, UUA, NOJ, SLY, LED, SGC]",False,3,1605,130,"[S7, SU, RT, YC, FV]","[2330, 1245]","[[1746617700, 1746750300], [1747198800, 174728...","[[SGC, UUA], [UUA, SGC]]",95ebc3ec5d902a8dc3256423d9905136,False,0.018526,0,4.152333,"[night_transfer, long_layover, overnight_layov...",SU
1930,"{'70': {'currency': 'rub', 'price': 107050, 'u...","{'70': {'VFLEXSNG,KECONALL,BOW;PSOW,YBD3,YBDST...",[{'flight': [{'aircraft': 'Boeing 737-500 (win...,3395,"[GSV, SVO, UUA, NOJ, SLY, SVX, SVO, SGC]",False,4,1210,110,"[UT, DP, RT, YC, SU]","[1965, 1430]","[[1746639600, 1746750300], [1747198800, 174729...","[[SGC, UUA], [UUA, SGC]]",1d777b3670e596ba5d9f401a740f59b1,False,0.035575,0,4.201667,"[long_layover, overnight_layover, airport_chan...",UT
1931,"{'70': {'currency': 'rub', 'price': 107050, 'u...","{'70': {'VFLEXSNG,KECONALL,BOW;PSOW,YBD3,YBDST...",[{'flight': [{'aircraft': 'Boeing 737-500 (win...,3395,"[GSV, SVO, UUA, NOJ, SLY, SVX, SVO, SGC]",False,4,890,110,"[UT, DP, RT, YC, SU]","[1965, 1430]","[[1746639600, 1746750300], [1747198800, 174729...","[[SGC, UUA], [UUA, SGC]]",30c8cfc99b052bf7a4699ab93e9f71f5,False,0.035575,0,4.201667,"[night_transfer, long_layover, overnight_layov...",UT
1932,"{'70': {'currency': 'rub', 'price': 107136, 'u...","{'70': {'TBSOW,XBSOW,BOW;PSOW,YBD3,YBDFL,HNOR'...",[{'flight': [{'air

In [9]:
proposals_df["baggage"]

,terms,xterms,segment,total_duration,stops_airports,is_charter,max_stops,max_stop_duration,min_stop_duration,carriers,segment_durations,segments_time,segments_airports,sign,is_direct,flight_weight,popularity,segments_rating,tags,validating_carrier
0,"{'70': {'currency': 'rub', 'price': 58761, 'un...","{'70': {'NLTROW,OLTOW,BOW;PSOW,YBSOWEM,HLTOW':...",[{'flight': [{'aircraft': 'Antonow / Antonov A...,6030,"[HMA, VKO, UUA, NOJ, OVB, SGC]",False,2,1595,815,"[UT, RT, S7]","[2630, 3400]","[[1746599700, 1746750300], [1747198800, 174741...","[[SGC, UUA], [UUA, SGC]]",e1f96b4b40dd91bf6808007ae3cb2363,False,0.035575,0,3.114000,"[overnight_layover, long_layover, has_night_tr...",UT
1,"{'70': {'currency': 'rub', 'price': 60134, 'un...","{'70': {'NLTROW,OLTOW,BOW;PSOW,YBSOWEM,XBSOW':...",[{'flight': [{'aircraft': 'Antonow / Antonov A...,5745,"[HMA, VKO, UUA, NOJ, OVB, SGC]",False,2,1435,815,"[S7, UT, RT]","[2630, 3115]","[[1746599700, 1746750300], [1747198800, 174739...","[[SGC, UUA], [UUA, SGC]]",08cb3828d30141a54ebc9ded2f3e05a7,False,0.035575,0,3.228000,"[has_night_transfer, night_transfer, long_layo...",UT
2,"{'70': {'currency': 'rub', 'price': 61013, 'un...","{'70': {'VLTOW,PLTOW,BOW;PSOW,YBSOWEM,HLTOW': ...",[{'flight': [{'aircraft': 'Sukhoi Superjet 100...,5340,"[SVX, SVO, UUA, NOJ, OVB, SGC]",False,2,1595,675,"[5N, RT, S7, UT, WZ]","[1940, 3400]","[[1746641100, 1746750300], [1747198800, 174741...","[[SGC, UUA], [UUA, SGC]]",32882d39e9ae3aaa94fbea70c4a32700,False,0.032666,0,3.418667,"[airport_change, has_night_transfer, night_tra...",5N
3,"{'70': {'currency': 'rub', 'price': 61391, 'un...","{'70': {'NLTROW,NNOR,BOW;PSOW,YBSOWEM,HLTOW': ...",[{'flight': [{'aircraft': 'Antonow / Antonov A...,6030,"[HMA, SVO, UUA, NOJ, OVB, SGC]",False,2,1595,870,"[UT, SU, RT, S7]","[2630, 3400]","[[1746599700, 1746750300], [1747198800, 174741...","[[SGC, UUA], [UUA, SGC]]",eea3d82cce78b7ebde00da013cee2b1d,False,0.018526,0,3.114000,"[long_layover, overnight_layover, airport_chan...",SU
4,"{'70': {'currency': 'rub', 'price': 61574, 'un...","{'70': {'NSTROW,OSTDOW,BOW;PSOW,YSTOWEM,TSTOW'...",[{'flight': [{'aircraft': 'Antonow / Antonov A...,5155,"[HMA, VKO, UUA, NOJ, OVB, SGC]",False,2,1435,765,"[UT, RT, S7]","[2630, 2525]","[[1746599700, 1746750300], [1747198800, 174735...","[[SGC, UUA], [UUA, SGC]]",1f48b5c2ea4f1d674cbd12bd2c2332d1,False,0.035575,0,3.443000,"[long_layover, overnight_layover, has_night_tr...",UT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1929,"{'70': {'currency': 'rub', 'price': 106276, 'u...","{'70': {'TBSOW,TNOR,BOW;PSOW,YBD3,YBDFL,HNOR':...",[{'flight': [{'aircraft': 'Embraer EMB 170 / E...,3575,"[OVB, SVO, UUA, NOJ, SLY, LED, SGC]",False,3,1605,130,"[S7, SU, RT, YC, FV]","[2330, 1245]","[[1746617700, 1746750300], [1747198800, 174728...","[[SGC, UUA], [UUA, SGC]]",95ebc3ec5d902a8dc3256423d9905136,False,0.018526,0,4.152333,"[night_transfer, long_layover, overnight_layov...",SU
1930,"{'70': {'currency': 'rub', 'price': 107050, 'u...","{'70': {'VFLEXSNG,KECONALL,BOW;PSOW,YBD3,YBDST...",[{'flight': [{'aircraft': 'Boeing 737-500 (win...,3395,"[GSV, SVO, UUA, NOJ, SLY, SVX, SVO, SGC]",False,4,1210,110,"[UT, DP, RT, YC, SU]","[1965, 1430]","[[1746639600, 1746750300], [1747198800, 174729...","[[SGC, UUA], [UUA, SGC]]",1d777b3670e596ba5d9f401a740f59b1,False,0.035575,0,4.201667,"[long_layover, overnight_layover, airport_chan...",UT
1931,"{'70': {'currency': 'rub', 'price': 107050, 'u...","{'70': {'VFLEXSNG,KECONALL,BOW;PSOW,YBD3,YBDST...",[{'flight': [{'aircraft': 'Boeing 737-500 (win...,3395,"[GSV, SVO, UUA, NOJ, SLY, SVX, SVO, SGC]",False,4,890,110,"[UT, DP, RT, YC, SU]","[1965, 1430]","[[1746639600, 1746750300], [1747198800, 174729...","[[SGC, UUA], [UUA, SGC]]",30c8cfc99b052bf7a4699ab93e9f71f5,False,0.035575,0,4.201667,"[night_transfer, long_layover, overnight_layov...",UT
1932,"{'70': {'currency': 'rub', 'price': 107136, 'u...","{'70': {'TBSOW,XBSOW,BOW;PSOW,YBD3,YBDFL,HNOR'...",[{'flight': [{'air

In [60]:
proposals[0]['terms']['70'].keys()

dict_keys(['currency', 'price', 'unified_price', 'url', 'transfer_terms', 'flight_additional_tariff_infos', 'flights_baggage', 'flights_handbags', 'baggage_source', 'handbags_source'])

In [44]:
proposals[-1]["max_stops"]

4

In [27]:
set(list(chain(*[i["tags"] for i in proposals])))


{'airport_change',
 'has_night_transfer',
 'long_layover',
 'night_transfer',
 'overnight_layover',
 'short_layover'}